# Ridge calibration of the ranking

Checks whether weights learned from the **195 human relevance labels** rank facilities better than the
current hand-set formula in `ranking.py`, using leave-one-query-out validation (train on 14 queries, test on 1).

- **Labels:** `datathon_195_labels_en.xlsx` (15 queries x 13 candidates, scored 0-3)
- **Features:** rebuilt from the current `facilities_clean` and `facility_service_evidence`, not the label-time snapshot
- **Baseline:** the production score from `ranking.py` (same location weights and decay; checked in `tests/test_train_ridge.py`)
- **Distance:** road distance from OSRM, the same as the app; estimated as 1.4 x straight line when unavailable
- All logic lives in `train_ridge.py`; this notebook only loads data and shows results

Run from the repository root of the Databricks Git folder.

In [0]:
%pip install openpyxl

In [0]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "app"))  # app/routing.py provides the OSRM road distances

import train_ridge as tr
from routing import road_km

LABEL_FILE = REPO_ROOT / "datathon_195_labels_en.xlsx"

labels = pd.read_excel(LABEL_FILE, sheet_name="manual_labels", header=2, engine="openpyxl")
snapshot = pd.read_excel(LABEL_FILE, sheet_name="training_features", header=2, engine="openpyxl")
labels = labels[["row_id", "label"]].dropna(subset=["row_id"])
snapshot = snapshot.dropna(subset=["row_id"])
assert set(labels["row_id"]) == set(snapshot["row_id"]), "Row IDs differ between the two sheets"

base = (snapshot[["row_id", "query_id", "facility_id", "service_code", "query_lat", "query_lon"]]
        .merge(labels, on="row_id", validate="one_to_one")
        .assign(label=lambda d: pd.to_numeric(d["label"], errors="raise").astype(int)))

evidence = (spark.table("workspace.datathon.facility_service_evidence")
            .select("unique_id", "service", "evidence_score", "evidence_level", "n_direct", "n_supporting")
            .toPandas())
facilities = (spark.table("workspace.datathon.facilities_clean")
              .select("unique_id", "facility_type", "n_quality_flags", "flag_capability_bloat",
                      "completeness", "location_confidence", "latitude", "longitude")
              .toPandas())

df = tr.build_features(base, evidence, facilities, distance_fn=road_km)
tr.check_training_data(df)
print(f"Ready: {len(df)} rows, {df['query_id'].nunique()} queries")
print("Distance basis:", df["distance_basis"].value_counts().to_dict())
print("Labels:", df["label"].value_counts().sort_index().to_dict())

In [0]:
# How much the evidence changed since the labels were drawn (the rules were updated in between)
old = snapshot[["row_id", "evidence_score", "n_direct"]].rename(
    columns={"evidence_score": "label_time_evidence_score", "n_direct": "label_time_n_direct"})
changed = (df[["row_id", "query_id", "service", "label", "evidence_score", "n_direct"]]
           .merge(old, on="row_id")
           .loc[lambda d: (d["evidence_score"] - d["label_time_evidence_score"]).abs() > 0.05]
           .sort_values("row_id"))
print(f"Rows whose evidence score changed by more than 0.05: {len(changed)} / {len(df)}")
display(changed)

In [0]:
per_query, df["ridge_test_score"] = tr.leave_one_query_out(df)
summary = per_query.groupby("model")[["ndcg_at_5", "precision_at_3"]].mean()
print("Average over the 15 held-out queries")
display(summary.round(3).reset_index())

wide = per_query.pivot(index="query_id", columns="model", values="ndcg_at_5")
delta = (wide["Ridge"] - wide["Current formula"]).round(6)
print(f"Ridge vs current formula on NDCG@5 - wins: {(delta > 0).sum()}, ties: {(delta == 0).sum()}, "
      f"losses: {(delta < 0).sum()}")
display(wide.assign(difference=delta).round(3).reset_index())

In [0]:
weights, intercept = tr.fit_weights(df)
print(f"Final model trained on all {len(df)} labelled candidates; intercept {intercept:.3f}")
display(weights.round(3))
warnings = tr.sign_warnings(weights)
if warnings:
    print("Check the labels before adopting these weights:")
    for w in warnings:
        print(" -", w)
else:
    print("Distance, evidence and location weights have the expected signs.")

### How to use the result
- **Adopt the Ridge weights only if they clearly beat the current formula** on held-out NDCG@5 and win more queries
  than they lose. Then `ranking.py` needs a scoring function that reads these weights, and the app's road-distance and
  AI-review layers must call it.
- **Otherwise keep the current formula** and report that the human labels support it.
- Caveats to state: 195 labels from 15 queries; only dialysis, CT and MRI (routine services) are labelled, so urgent
  services are not covered; labels were scored by the team, partly with AI suggestions.